# Pipeline: Social Media → Donor Engagement

## Problem framing
**Who cares**: Founder / fundraising lead / social media manager.

**Business question**: Which supporters are most likely to donate soon — and what social media content should we put in front of them to make it happen?

**What makes this different from the other pipelines**:
- The **donor lapse pipeline** predicts *who* will lapse, but doesn't know *why* or what content resonates.
- The **outreach pipeline** predicts post-level donation value, but doesn't connect to specific supporters.
- This pipeline **merges both**: supporter behavior (RFM) + social media context (what posts preceded their donations, what campaigns were running) to predict who will donate next AND what content drives it.

## Data linkages used
- `donations.referral_post_id` → `social_media_posts.post_id` (77 direct links)
- `donations.campaign_name` ↔ `social_media_posts.campaign_name` (4 shared campaigns)
- `supporters.acquisition_channel` = "SocialMedia" (13 supporters)
- `donations.channel_source` = "SocialMedia" (78 donations)

## Output
- `predictions_social_media_donor_engagement.csv`
- Content strategy recommendations based on feature importance

In [1]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pd.set_option("display.max_columns", 200)

HERE = Path.cwd()
if (HERE / "donations.csv").exists():
    DATA_DIR = HERE
elif (HERE.parent / "donations.csv").exists():
    DATA_DIR = HERE.parent
else:
    raise FileNotFoundError("Could not locate donations.csv")

supporters = pd.read_csv(DATA_DIR / "supporters.csv")
donations = pd.read_csv(DATA_DIR / "donations.csv")
posts = pd.read_csv(DATA_DIR / "social_media_posts.csv")

donations["donation_date"] = pd.to_datetime(donations["donation_date"], errors="coerce")
donations["estimated_value"] = pd.to_numeric(donations["estimated_value"], errors="coerce")
posts["created_at"] = pd.to_datetime(posts["created_at"], errors="coerce")

print(f"Supporters: {len(supporters)}, Donations: {len(donations)}, Posts: {len(posts)}")
print(f"Donations with referral_post_id: {donations['referral_post_id'].notna().sum()}")

Supporters: 60, Donations: 420, Posts: 812
Donations with referral_post_id: 77


In [2]:
# Build snapshot dataset: one row per supporter per month
# Features come from BEFORE the snapshot; label comes from AFTER.

HORIZON_DAYS = 90
SM_LOOKBACK_DAYS = 60

last_month = donations["donation_date"].max().to_period("M").to_timestamp()
first_month = donations["donation_date"].min().to_period("M").to_timestamp()
months = pd.date_range(first_month, last_month, freq="MS")

snapshots = pd.MultiIndex.from_product(
    [supporters["supporter_id"].unique(), months],
    names=["supporter_id", "snapshot_date"],
).to_frame(index=False)

print(f"Raw snapshot grid: {len(snapshots)} rows ({len(supporters)} supporters × {len(months)} months)")

Raw snapshot grid: 2340 rows (60 supporters × 39 months)


In [3]:
# === FEATURE BLOCK 1: Supporter RFM features ===
# Computed from donation history strictly before each snapshot date.

don = donations.copy()
don["is_recurring"] = don.get("is_recurring", pd.Series(dtype=bool)).fillna(False).astype(bool)

def build_rfm(snap_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for sid, sdate in snap_df[["supporter_id", "snapshot_date"]].itertuples(index=False):
        hist = don[(don["supporter_id"] == sid) & (don["donation_date"] < sdate)]
        if hist.empty:
            rows.append((sid, sdate, np.nan, 0, 0.0, 0.0, 0, 0, 0, None, None))
            continue
        recency = (sdate - hist["donation_date"].max()).days
        tenure = (sdate - hist["donation_date"].min()).days
        freq = len(hist)
        value_sum = float(hist["estimated_value"].fillna(0).sum())
        value_avg = float(hist["estimated_value"].fillna(0).mean())
        recurring_n = int(hist["is_recurring"].sum())
        unique_campaigns = int(hist["campaign_name"].nunique())
        sm_channel_n = int((hist["channel_source"] == "SocialMedia").sum())
        top_channel = hist["channel_source"].mode().iloc[0] if hist["channel_source"].notna().any() else None
        top_campaign = hist["campaign_name"].mode().iloc[0] if hist["campaign_name"].notna().any() else None
        rows.append((sid, sdate, recency, freq, value_sum, value_avg,
                      recurring_n, unique_campaigns, sm_channel_n, top_channel, top_campaign))

    return pd.DataFrame(rows, columns=[
        "supporter_id", "snapshot_date",
        "recency_days", "donation_frequency", "donation_value_sum", "donation_value_avg",
        "recurring_count", "unique_campaigns", "sm_channel_donations",
        "top_channel_source", "top_campaign",
    ])

rfm = build_rfm(snapshots)
rfm = rfm[rfm["donation_frequency"] > 0].copy()
print(f"RFM features built: {len(rfm)} supporter-month rows (with ≥1 prior donation)")
rfm.head()

RFM features built: 1872 supporter-month rows (with ≥1 prior donation)


,supporter_id,snapshot_date,recency_days,donation_frequency,donation_value_sum,donation_value_avg,recurring_count,unique_campaigns,sm_channel_donations,top_channel_source,top_campaign
3,1,2023-04-01,7.0,1,774.61,774.610,1,0,1,SocialMedia,None
4,1,2023-05-01,37.0,1,774.61,774.610,1,0,1,SocialMedia,None
5,1,2023-06-01,68.0,1,774.61,774.610,1,0,1,SocialMedia,None
6,1,2023-07-01,7.0,2,1381.52,690.760,2,1,1,Direct,Back to School
7,1,2023-08-01,9.0,5,2364.66,472.932,5,1,2,Event,Back to School


In [4]:
# === FEATURE BLOCK 2: Social media affinity per supporter ===
# How connected is this supporter to social media content?
# Based on their historical referral-linked donations.

ref_donations = don[don["referral_post_id"].notna()].copy()
ref_merged = ref_donations.merge(posts, left_on="referral_post_id", right_on="post_id", how="inner")

def build_sm_affinity(snap_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for sid, sdate in snap_df[["supporter_id", "snapshot_date"]].itertuples(index=False):
        hist_ref = ref_merged[(ref_merged["supporter_id"] == sid) & (ref_merged["donation_date"] < sdate)]
        hist_all = don[(don["supporter_id"] == sid) & (don["donation_date"] < sdate)]

        total_donations = len(hist_all)
        referral_linked = len(hist_ref)
        referral_pct = referral_linked / total_donations if total_donations > 0 else 0.0

        if hist_ref.empty:
            rows.append((sid, sdate, referral_linked, referral_pct, None, None, None, None, 0))
            continue

        top_platform = hist_ref["platform"].mode().iloc[0] if hist_ref["platform"].notna().any() else None
        top_topic = hist_ref["content_topic"].mode().iloc[0] if hist_ref["content_topic"].notna().any() else None
        top_cta = hist_ref["call_to_action_type"].mode().iloc[0] if hist_ref["call_to_action_type"].notna().any() else None
        top_media = hist_ref["media_type"].mode().iloc[0] if hist_ref["media_type"].notna().any() else None
        story_posts = int(hist_ref["features_resident_story"].fillna(False).astype(bool).sum())

        rows.append((sid, sdate, referral_linked, referral_pct,
                      top_platform, top_topic, top_cta, top_media, story_posts))

    return pd.DataFrame(rows, columns=[
        "supporter_id", "snapshot_date",
        "referral_linked_donations", "referral_pct",
        "preferred_platform", "preferred_topic", "preferred_cta", "preferred_media_type",
        "story_post_donations",
    ])

sm_affinity = build_sm_affinity(rfm[["supporter_id", "snapshot_date"]])
print(f"SM affinity features: {len(sm_affinity)} rows")
print(f"Supporters with ≥1 referral-linked donation: {(sm_affinity['referral_linked_donations'] > 0).sum()}")

SM affinity features: 1872 rows
Supporters with ≥1 referral-linked donation: 937


In [5]:
# === FEATURE BLOCK 3: Period social media activity ===
# What was happening on social media in the SM_LOOKBACK_DAYS before each snapshot?
# This captures the "content environment" that could trigger a donation.

def build_sm_period(snap_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for sid, sdate in snap_df[["supporter_id", "snapshot_date"]].itertuples(index=False):
        window_start = sdate - pd.Timedelta(days=SM_LOOKBACK_DAYS)
        recent_posts = posts[(posts["created_at"] >= window_start) & (posts["created_at"] < sdate)]

        n_posts = len(recent_posts)
        if n_posts == 0:
            rows.append((sid, sdate, 0, 0.0, 0.0, 0.0, 0, 0, 0, 0.0, None, None, None, 0))
            continue

        avg_engagement = float(recent_posts["engagement_rate"].mean())
        avg_referrals = float(recent_posts["donation_referrals"].fillna(0).mean())
        total_referrals = float(recent_posts["donation_referrals"].fillna(0).sum())
        boosted_n = int(recent_posts["is_boosted"].fillna(False).astype(bool).sum())
        story_n = int(recent_posts["features_resident_story"].fillna(False).astype(bool).sum())
        cta_n = int(recent_posts["has_call_to_action"].fillna(False).astype(bool).sum())
        avg_reach = float(recent_posts["reach"].fillna(0).mean())

        dominant_platform = recent_posts["platform"].mode().iloc[0] if recent_posts["platform"].notna().any() else None
        dominant_topic = recent_posts["content_topic"].mode().iloc[0] if recent_posts["content_topic"].notna().any() else None
        active_campaign = recent_posts["campaign_name"].mode().iloc[0] if recent_posts["campaign_name"].notna().any() else None
        campaign_active = int(recent_posts["campaign_name"].notna().any())

        rows.append((sid, sdate, n_posts, avg_engagement, avg_referrals, total_referrals,
                      boosted_n, story_n, cta_n, avg_reach,
                      dominant_platform, dominant_topic, active_campaign, campaign_active))

    return pd.DataFrame(rows, columns=[
        "supporter_id", "snapshot_date",
        "sm_posts_last_60d", "sm_avg_engagement", "sm_avg_referrals", "sm_total_referrals",
        "sm_boosted_posts", "sm_story_posts", "sm_cta_posts", "sm_avg_reach",
        "sm_dominant_platform", "sm_dominant_topic", "sm_active_campaign", "sm_campaign_active",
    ])

sm_period = build_sm_period(rfm[["supporter_id", "snapshot_date"]])
print(f"Period SM features: {len(sm_period)} rows")
sm_period.head()

Period SM features: 1872 rows


,supporter_id,snapshot_date,sm_posts_last_60d,sm_avg_engagement,sm_avg_referrals,sm_total_referrals,sm_boosted_posts,sm_story_posts,sm_cta_posts,sm_avg_reach,sm_dominant_platform,sm_dominant_topic,sm_active_campaign,sm_campaign_active
0,1,2023-04-01,41,0.102293,14.780488,606.0,8,10,24,4326.073171,Instagram,Education,None,0
1,1,2023-05-01,47,0.109432,14.553191,684.0,10,10,29,4357.255319,Instagram,SafehouseLife,Summer of Safety,1
2,1,2023-06-01,46,0.102352,15.369565,707.0,9,10,32,3902.891304,Instagram,DonorImpact,Summer of Safety,1
3,1,2023-07-01,32,0.085169,12.406250,397.0,6,8,21,3785.437500,Instagram,Education,Summer of Safety,1
4,1,2023-08-01,40,0.084550,5.225000,209.0,3,6,26,2677.825000,Facebook,SafehouseLife,Back to School,1


In [6]:
# === MERGE ALL FEATURES + LABEL ===

# Label: will this supporter donate in the next 90 days?
labels = []
for sid, sdate in rfm[["supporter_id", "snapshot_date"]].itertuples(index=False):
    future = don[(don["supporter_id"] == sid)
                 & (don["donation_date"] >= sdate)
                 & (don["donation_date"] < sdate + pd.Timedelta(days=HORIZON_DAYS))]
    labels.append((sid, sdate, int(len(future) > 0)))

label_df = pd.DataFrame(labels, columns=["supporter_id", "snapshot_date", "will_donate_next_90d"])

# Static supporter attributes
static_cols = ["supporter_type", "relationship_type", "region", "country", "acquisition_channel", "status"]
static_cols = [c for c in static_cols if c in supporters.columns]
static = supporters[["supporter_id"] + static_cols].copy()

# Merge everything
model_df = (
    rfm
    .merge(sm_affinity, on=["supporter_id", "snapshot_date"], how="left")
    .merge(sm_period, on=["supporter_id", "snapshot_date"], how="left")
    .merge(label_df, on=["supporter_id", "snapshot_date"], how="left")
    .merge(static, on="supporter_id", how="left")
)

# Derived features
model_df["is_sm_acquired"] = (model_df["acquisition_channel"] == "SocialMedia").astype(int)
model_df["sm_donation_ratio"] = model_df["sm_channel_donations"] / model_df["donation_frequency"].clip(lower=1)

print(f"Final model dataset: {len(model_df)} rows, {model_df.shape[1]} columns")
print(f"Label balance: {model_df['will_donate_next_90d'].mean():.2%} positive")
model_df.head()

Final model dataset: 1872 rows, 39 columns
Label balance: 39.00% positive


,supporter_id,snapshot_date,recency_days,donation_frequency,donation_value_sum,donation_value_avg,recurring_count,unique_campaigns,sm_channel_donations,top_channel_source,top_campaign,referral_linked_donations,referral_pct,preferred_platform,preferred_topic,preferred_cta,preferred_media_type,story_post_donations,sm_posts_last_60d,sm_avg_engagement,sm_avg_referrals,sm_total_referrals,sm_boosted_posts,sm_story_posts,sm_cta_posts,sm_avg_reach,sm_dominant_platform,sm_dominant_topic,sm_active_campaign,sm_campaign_active,will_donate_next_90d,supporter_type,relationship_type,region,country,acquisition_channel,status,is_sm_acquired,sm_donation_ratio
0,1,2023-04-01,7.0,1,774.61,774.610,1,0,1,SocialMedia,None,1,1.0,Instagram,Gratitude,LearnMore,Video,1,41,0.102293,14.780488,606.0,8,10,24,4326.073171,Instagram,Education,None,0,1,SocialMediaAdvocate,Local,Luzon,Philippines,SocialMedia,Active,1,1.0
1,1,2023-05-01,37.0,1,774.61,774.610,1,0,1,SocialMedia,None,1,1.0,Instagram,Gratitude,LearnMore,Video,1,47,0.109432,14.553191,684.0,10,10,29,4357.255319,Instagram,SafehouseLife,Summer of Safety,1,1,SocialMediaAdvocate,Local,Luzon,Philippines,SocialMedia,Active,1,1.0
2,1,2023-06-01,68.0,1,774.61,774.610,1,0,1,SocialMedia,None,1,1.0,Instagram,Gratitude,LearnMore,Video,1,46,0.102352,15.369565,707.0,9,10,32,3902.891304,Instagram,DonorImpact,Summer of Safety,1,1,SocialMediaAdvocate,Local,Luzon,Philippines,SocialMedia,Active,1,1.0
3,1,2023-07-01,7.0,2,1381.52,690.760,2,1,1,Direct,Back to School,1,0.5,Instagram,Gratitude,LearnMore,Video,1,32,0.085169,12.406250,397.0,6,8,21,3785.437500,Instagram,Education,Summer of Safety,1,1,SocialMediaAdvocate,Local,Luzon,Philippines,SocialMedia,Active,1,0.5
4,1,2023-08-01,9.0,5,2364.66,472.932,5,1,2,Event,Back to School,2,0.4,Instagram,CampaignLaunch,LearnMore,Reel,2,40,0.084550,5.225000,209.0,3,6,26,2677.825000,Facebook,SafehouseLife,Back to School,1,0,SocialMediaAdvocate,Local,Luzon,Philippines,SocialMedia,Active,1,0.4


In [7]:
# === MODEL TRAINING ===

TARGET = "will_donate_next_90d"
exclude = {"supporter_id", "snapshot_date", TARGET}
feature_cols = [c for c in model_df.columns if c not in exclude]

X_all = model_df[feature_cols].copy()
y_all = model_df[TARGET].astype(int)

can_stratify = y_all.nunique() > 1 and y_all.value_counts().min() >= 2
X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.2, random_state=42,
    stratify=y_all if can_stratify else None,
)

numeric_cols = X_train.select_dtypes(include=["number"]).columns.tolist()
cat_cols = [c for c in X_train.columns if c not in numeric_cols]

for c in cat_cols:
    X_train[c] = X_train[c].astype("object")
    X_test[c] = X_test[c].astype("object")

pre = ColumnTransformer(
    [
        ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), numeric_cols),
        ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")), ("ohe", OneHotEncoder(handle_unknown="ignore"))]), cat_cols),
    ]
)

models = {
    "LogisticRegression": LogisticRegression(max_iter=3000, class_weight="balanced", random_state=42),
    "RandomForest": RandomForestClassifier(n_estimators=300, max_depth=12, class_weight="balanced", random_state=42, n_jobs=-1),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
best_name, best_score, best_pipe = None, -1e9, None

for name, model in models.items():
    p = Pipeline([("pre", pre), ("model", model)])
    scores = cross_val_score(p, X_train, y_train, cv=cv, scoring="roc_auc")
    print(f"{name}: CV ROC-AUC = {scores.mean():.3f} ± {scores.std():.3f}")
    if scores.mean() > best_score:
        best_name, best_score, best_pipe = name, scores.mean(), p

print(f"\nBest model: {best_name}")
best_pipe.fit(X_train, y_train)
pipe = best_pipe

proba = pipe.predict_proba(X_test)[:, 1]
pred = (proba >= 0.5).astype(int)

print(f"\n--- Test Set Results ---")
print(f"ROC-AUC:  {roc_auc_score(y_test, proba):.3f}")
print(f"PR-AUC:   {average_precision_score(y_test, proba):.3f}")
print(classification_report(y_test, pred, zero_division=0))

LogisticRegression: CV ROC-AUC = 0.691 ± 0.017


RandomForest: CV ROC-AUC = 0.802 ± 0.026

Best model: RandomForest

--- Test Set Results ---
ROC-AUC:  0.841
PR-AUC:   0.766
              precision    recall  f1-score   support

           0       0.77      0.90      0.83       229
           1       0.79      0.58      0.67       146

    accuracy                           0.78       375
   macro avg       0.78      0.74      0.75       375
weighted avg       0.78      0.78      0.77       375



In [8]:
# === FEATURE IMPORTANCE ===
# Which features matter most for predicting donation engagement?

ohe = pipe.named_steps["pre"].named_transformers_["cat"].named_steps["ohe"]
cat_feature_names = ohe.get_feature_names_out(cat_cols).tolist() if len(cat_cols) else []
feat_names = numeric_cols + cat_feature_names

model = pipe.named_steps["model"]
if hasattr(model, "coef_"):
    imp = model.coef_[0]
    label = "coef"
else:
    imp = model.feature_importances_
    label = "importance"

imp_df = pd.DataFrame({"feature": feat_names, label: imp}).sort_values(label, key=abs, ascending=False)

print(f"Top 25 features by |{label}| ({best_name})")
print("=" * 60)

# Group features into categories for readability
for _, row in imp_df.head(25).iterrows():
    feat = row["feature"]
    val = row[label]
    if feat.startswith("sm_"):
        category = "📱 Social Media"
    elif feat.startswith("referral_") or feat.startswith("preferred_") or feat == "story_post_donations":
        category = "🔗 SM Affinity"
    elif feat.startswith("donation_") or feat in ("recency_days", "recurring_count", "unique_campaigns"):
        category = "💰 Donor History"
    else:
        category = "📋 Other"
    print(f"  {category:20s}  {feat:40s}  {val:+.4f}")

display(imp_df.head(25))

Top 25 features by |importance| (RandomForest)
  💰 Donor History       recency_days                              +0.0646
  💰 Donor History       donation_value_sum                        +0.0592
  💰 Donor History       donation_value_avg                        +0.0502
  💰 Donor History       recurring_count                           +0.0473
  📱 Social Media        sm_avg_referrals                          +0.0405
  💰 Donor History       donation_frequency                        +0.0343
  📱 Social Media        sm_total_referrals                        +0.0332
  📱 Social Media        sm_cta_posts                              +0.0314
  📱 Social Media        sm_story_posts                            +0.0310
  📱 Social Media        sm_avg_reach                              +0.0290
  📱 Social Media        sm_avg_engagement                         +0.0290
  📱 Social Media        sm_posts_last_60d                         +0.0266
  📱 Social Media        sm_boosted_posts                         

,feature,importance
0,recency_days,0.064584
2,donation_value_sum,0.059207
3,donation_value_avg,0.050182
4,recurring_count,0.047272
12,sm_avg_referrals,0.040524
1,donation_frequency,0.034255
13,sm_total_referrals,0.033219
16,sm_cta_posts,0.031436
15,sm_story_posts,0.031005
17,sm_avg_reach,0.029032


In [9]:
# === GENERATE ACTIONABLE PREDICTIONS ===

latest = model_df.sort_values(["supporter_id", "snapshot_date"]).groupby("supporter_id").tail(1).copy()

X_latest = latest[feature_cols].copy()
for c in cat_cols:
    if c in X_latest.columns:
        X_latest[c] = X_latest[c].astype("object")

latest["engagement_probability"] = pipe.predict_proba(X_latest)[:, 1]

# Risk tiers — percentile-based so the output is always actionable
# Top 25% = High, middle 40% = Medium, bottom 35% = Low
p75 = latest["engagement_probability"].quantile(0.75)
p35 = latest["engagement_probability"].quantile(0.35)

def risk_tier(p):
    if p >= p75:
        return "High"
    elif p >= p35:
        return "Medium"
    else:
        return "Low"

latest["engagement_tier"] = latest["engagement_probability"].apply(risk_tier)
print(f"Tier thresholds: High ≥ {p75:.3f}, Medium ≥ {p35:.3f}, Low < {p35:.3f}")

# Suggested action based on tier + SM affinity
def suggest_action(row):
    tier = row["engagement_tier"]
    sm_ratio = row.get("sm_donation_ratio", 0) or 0
    referral_n = row.get("referral_linked_donations", 0) or 0

    if tier == "High":
        if sm_ratio > 0.3:
            return "Send targeted SM campaign — this donor responds to social media"
        return "Send personalized thank-you + campaign update"
    elif tier == "Medium":
        if referral_n > 0:
            return "Re-engage with content similar to posts that drove prior donations"
        return "Include in next campaign email + boost SM posts in their region"
    else:
        if sm_ratio > 0.3:
            return "Try a direct SM message or story tag — they engage digitally"
        return "Personal outreach (call/meeting) — at risk of lapsing"

latest["suggested_action"] = latest.apply(suggest_action, axis=1)

# Join display names
latest = latest.merge(
    supporters[["supporter_id", "display_name", "email", "first_name", "last_name"]],
    on="supporter_id",
    how="left",
)

out_cols = [
    "supporter_id", "display_name", "email",
    "engagement_tier", "engagement_probability", "suggested_action",
    "recency_days", "donation_frequency", "donation_value_sum",
    "sm_channel_donations", "referral_linked_donations", "referral_pct",
    "preferred_platform", "preferred_topic", "preferred_cta",
    "sm_posts_last_60d", "sm_active_campaign",
    "acquisition_channel", "supporter_type", "region", "country",
]
out_cols = [c for c in out_cols if c in latest.columns]

out = latest[out_cols].sort_values("engagement_probability", ascending=False).copy()

out_path = DATA_DIR / "predictions_social_media_donor_engagement.csv"
out.to_csv(out_path, index=False)

print(f"Saved: {out_path}")
print(f"\nEngagement tier distribution:")
print(out["engagement_tier"].value_counts().to_string())
print(f"\n--- Top 15 supporters by engagement likelihood ---")
display(out.head(15))

Tier thresholds: High ≥ 0.075, Medium ≥ 0.044, Low < 0.044
Saved: /Users/masonzarges/Desktop/INTEX Data/predictions_social_media_donor_engagement.csv

Engagement tier distribution:
engagement_tier
Medium    23
Low       21
High      15

--- Top 15 supporters by engagement likelihood ---


,supporter_id,display_name,email,engagement_tier,engagement_probability,suggested_action,recency_days,donation_frequency,donation_value_sum,sm_channel_donations,referral_linked_donations,referral_pct,preferred_platform,preferred_topic,preferred_cta,sm_posts_last_60d,sm_active_campaign,acquisition_channel,supporter_type,region,country
3,4,Liam Diaz,liam-diaz@globe.com.ph,High,0.652772,Send personalized thank-you + campaign update,97.0,10,9591.66,3,3,0.300000,Facebook,SafehouseLife,DonateNow,52,None,Church,MonetaryDonor,Mindanao,Philippines
8,9,Sophia Ibrahim,sophia-ibrahim@globe.com.ph,High,0.220142,Send targeted SM campaign — this donor respond...,26.0,9,10315.75,4,4,0.444444,Instagram,DonorImpact,LearnMore,52,None,PartnerReferral,Volunteer,Luzon,Philippines
46,48,Ian Xu,ian-xu@pldt.net.ph,High,0.151932,Send personalized thank-you + campaign update,101.0,4,2257.28,0,0,0.000000,None,None,None,52,None,PartnerReferral,SkillsContributor,Luzon,Philippines
23,24,Sean Xu,sean-xu@yahoo.com.ph,High,0.150793,Send personalized thank-you + campaign update,138.0,9,4168.88,1,1,0.111111,YouTube,SafehouseLife,DonateNow,52,None,SocialMedia,MonetaryDonor,Visayas,Philippines
24,25,Tina Young,tina-young@yahoo.com.ph,High,0.146949,Send personalized thank-you + campaign update,15.0,17,11540.99,1,1,0.058824,TikTok,CampaignLaunch,LearnMore,52,None,Church,MonetaryDonor,Mindanao,Philippines
25,26,Margo Zhang,margo-zhang@smart.com.ph,High,0.141315,Send personalized thank-you + campaign update,5.0,12,7109.62,3,3,0.250000,Facebook,DonorImpact,DonateNow,52,None,WordOfMouth,MonetaryDonor,Visayas,Philippines
29,31,Jon Edwards,jon-edwards@eastern.com.ph,High,0.133825,Send personalized thank-you + campaign update,18.0,20,10003.36,5,5,0.250000,YouTube,DonorImpact,DonateNow,52,None,Website,MonetaryDonor,Mindanao,Philippines
10,11,Maya Kim,maya-kim@gmail.com,High,0.126857,Send targeted SM campaign — this donor respond...,75.0,4,4395.62,3,3,0.750000,Facebook,SafehouseLife,DonateNow,52,None,SocialMedia,MonetaryDonor,Luzon,USA
58,60,Tess Khan,tess-khan@smart.com.ph,High,0.119641,Send personalized thank-you + campaign update,276.0,6,3976.01,0,0,0.000000,None,None,None,52,None,Website,InKindDonor,Visayas,Philippines
36,38,Faith Partners,faith-partners@faithpartners.ph,High,0.115247,Send personalized thank-you + campaign update,48.0,9,6792.70,2,2,0.222222,Twitter,AwarenessRaising,ShareStory,52,None,WordOfMouth,PartnerOrganization,Mindanao,Philippines


In [10]:
# === CONTENT STRATEGY RECOMMENDATIONS ===
# Based on the model's feature importances and the data patterns.

print("=" * 70)
print("CONTENT STRATEGY RECOMMENDATIONS")
print("Based on model feature importance + data patterns")
print("=" * 70)

# Analyze which SM features are most important
sm_features = imp_df[imp_df["feature"].str.contains("sm_|referral_|preferred_|story_|campaign")].head(15)

print("\n1. SOCIAL MEDIA FEATURES THAT MATTER MOST FOR DONATIONS:")
for _, row in sm_features.iterrows():
    print(f"   • {row['feature']}: {row[label]:+.4f}")

# Analyze which platforms/topics appear in high-engagement donors
high_eng = out[out["engagement_tier"] == "High"]
low_eng = out[out["engagement_tier"] == "Low"]

print(f"\n2. HIGH vs LOW ENGAGEMENT DONOR PROFILES:")
print(f"   High-engagement donors ({len(high_eng)}):")
if len(high_eng) > 0:
    print(f"     Avg donations: {high_eng['donation_frequency'].mean():.1f}")
    print(f"     Avg total value: {high_eng['donation_value_sum'].mean():.0f} PHP")
    print(f"     SM-linked donations: {high_eng['sm_channel_donations'].mean():.1f}")
    print(f"     Referral-linked: {high_eng['referral_linked_donations'].mean():.1f}")
    if high_eng["preferred_platform"].notna().any():
        print(f"     Top platforms: {high_eng['preferred_platform'].value_counts().head(3).to_dict()}")
    if high_eng["preferred_topic"].notna().any():
        print(f"     Top topics: {high_eng['preferred_topic'].value_counts().head(3).to_dict()}")

print(f"\n   Low-engagement donors ({len(low_eng)}):")
if len(low_eng) > 0:
    print(f"     Avg donations: {low_eng['donation_frequency'].mean():.1f}")
    print(f"     Avg total value: {low_eng['donation_value_sum'].mean():.0f} PHP")
    print(f"     SM-linked donations: {low_eng['sm_channel_donations'].mean():.1f}")
    print(f"     Referral-linked: {low_eng['referral_linked_donations'].mean():.1f}")

# Actionable summary
print("\n3. WHAT TO DO NEXT:")
print("   → For HIGH-tier supporters: maintain engagement with personalized campaigns")
print("   → For MEDIUM-tier: increase SM touchpoints with content matching their preferred topic/platform")
print("   → For LOW-tier: prioritize direct outreach (calls, meetings) before they lapse completely")
print("   → Check the 'suggested_action' column in the CSV for per-supporter recommendations")

CONTENT STRATEGY RECOMMENDATIONS
Based on model feature importance + data patterns

1. SOCIAL MEDIA FEATURES THAT MATTER MOST FOR DONATIONS:
   • sm_avg_referrals: +0.0405
   • sm_total_referrals: +0.0332
   • sm_cta_posts: +0.0314
   • sm_story_posts: +0.0310
   • sm_avg_reach: +0.0290
   • sm_avg_engagement: +0.0290
   • sm_posts_last_60d: +0.0266
   • sm_boosted_posts: +0.0255
   • referral_pct: +0.0205
   • sm_donation_ratio: +0.0201
   • unique_campaigns: +0.0175
   • sm_channel_donations: +0.0106
   • sm_dominant_topic_CampaignLaunch: +0.0096
   • referral_linked_donations: +0.0096
   • top_campaign_Back to School: +0.0086

2. HIGH vs LOW ENGAGEMENT DONOR PROFILES:
   High-engagement donors (15):
     Avg donations: 9.1
     Avg total value: 6733 PHP
     SM-linked donations: 1.6
     Referral-linked: 1.6
     Top platforms: {'Facebook': 3, 'Instagram': 2, 'YouTube': 2}
     Top topics: {'SafehouseLife': 4, 'DonorImpact': 3, 'AwarenessRaising': 2}

   Low-engagement donors (21):
